# Practical 2: Image Classification with a Pre-trained Model (Fine-tuning, MobileNetV2)

**Aim:** To classify flower images into 5 classes using transfer learning: reuse a MobileNetV2 network pre-trained on ImageNet, first as a fixed feature extractor and then fine-tune its top layers.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Validation accuracy ~87-90%.; Use preprocess_input() to scale pixels to [-1, 1]; use GlobalAveragePooling2D instead of Flatten to reduce parameters.

## Cell 1 - Import libraries

**What it does:**
- `MobileNetV2` is the pre-trained model. `preprocess_input` scales pixels exactly the way MobileNetV2 was trained (to [-1, 1]).

*Expected output (illustrative):* TensorFlow version: 2.x.x

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## Cell 2 - Download / locate the dataset

**What it does:**
- `get_file` downloads and extracts the archive once and returns its folder path.
- Each sub-folder name (daisy, roses, ...) becomes a class label automatically in the next cell.
- To use your own folder, replace this cell with `data_dir = pathlib.Path('path/to/flowers')`.

*Expected output (illustrative):* Total images: 3670 (for the standard flower_photos dataset).

In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = keras.utils.get_file("flower_photos", origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

image_count = len(list(data_dir.glob("*/*.jpg")))
print("Dataset folder:", data_dir)
print("Total images  :", image_count)

## Cell 3 - Create train / validation / test datasets

**What it does:**
- `image_dataset_from_directory` reads images from class sub-folders, resizes them to 224x224 and batches them.
- 80% of images go to training; the other 20% are loaded once into NumPy arrays and split in half: first half = validation (used while training), second half = test (final evaluation). Splitting fixed arrays guarantees the two sets never overlap and keep a stable order.
- `seed=123` makes the split reproducible - both calls must use the same seed so train and validation never overlap.
- `class_names` is the alphabetical list of folder names; `num_classes` = 5.

*Expected output (illustrative):* Found 3670 files belonging to 5 classes. Using 2936 files for training. Using 734 files for validation.
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Validation images: 367  Test images: 367

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training",
    seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

val_full = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation",
    seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

val_x = np.concatenate([x.numpy() for x, _ in val_full])
val_y = np.concatenate([y.numpy() for _, y in val_full])
half = len(val_x) // 2

val_ds = tf.data.Dataset.from_tensor_slices((val_x[:half], val_y[:half])).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((val_x[half:], val_y[half:])).batch(BATCH_SIZE)
print("Validation images:", half, " Test images:", len(val_x) - half)

## Cell 4 - Preprocess and optimise the input pipeline

**What it does:**
- `preprocess_input` converts pixel values 0-255 to the range [-1, 1] that MobileNetV2 expects. Using the wrong scaling hurts accuracy badly.
- `cache` keeps images in memory after the first epoch; `prefetch` prepares the next batch while the GPU trains; `shuffle` randomises the training order.

*Expected output (illustrative):* No output.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(1000)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

## Cell 5 - Data augmentation layer

**What it does:**
- Randomly flips, rotates (+/-10%) and zooms (+/-10%) training images so the model sees more variety and overfits less.
- These layers are active only during training; they do nothing at prediction time.

*Expected output (illustrative):* No output.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## Cell 6 - Build the model (frozen base + new head)

**What it does:**
- `weights='imagenet'` loads the pre-trained weights. `include_top=False` removes the original 1000-class classifier so we can attach our own.
- `base_model.trainable = False` **freezes** all 2.2 M base weights (Stage 1: feature extraction).
- `training=False` keeps BatchNormalization layers in inference mode - important when fine-tuning later.
- `GlobalAveragePooling2D` turns the 7x7x1280 feature map into a 1280-vector (far fewer parameters than Flatten).
- `Dense(5, softmax)` is the new head that outputs the 5 flower probabilities.

*Expected output (illustrative):* Output of the base: (None, 7, 7, 1280). After pooling: (None, 1280). Final output: (None, 5). Trainable params ~6,405 (1280*5 + 5); non-trainable ~2.26 M.

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

## Cell 7 - Stage 1: train only the new head

**What it does:**
- lr = 1e-3 is fine because only the small new head is being trained.
- Expect validation accuracy to rise quickly (MobileNetV2 features are already very good).

*Expected output (illustrative):* 5 epochs of logs; validation accuracy typically climbs into the high 80s.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

initial_epochs = 5
history1 = model.fit(train_ds, validation_data=val_ds, epochs=initial_epochs)

## Cell 8 - Stage 2: fine-tune the top 30 layers

**What it does:**
- The base is unfrozen, then everything except the **last 30 layers** is frozen again - only high-level features are adapted to flowers.
- The model **must be re-compiled** after changing `trainable`, otherwise the change has no effect.
- A very small learning rate (1e-5) makes gentle adjustments and avoids catastrophic forgetting.
- `initial_epoch` continues the epoch counter so the graphs join smoothly.

*Expected output (illustrative):* 5 more epochs (6/10 ... 10/10). Validation accuracy improves slightly, ~87-90% (from your PDF).

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_epochs = 5
total_epochs = initial_epochs + fine_tune_epochs
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=total_epochs, initial_epoch=history1.epoch[-1] + 1)

## Cell 9 - Plot combined training curves

**What it does:**
- Joins the Stage-1 and Stage-2 histories into one curve. The dashed line marks where fine-tuning begins.

*Expected output (illustrative):* Two graphs with 10 points each and a dashed vertical line after epoch 5.

In [ ]:
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss = history1.history["loss"] + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label="Train")
plt.plot(val_acc, label="Validation")
plt.axvline(initial_epochs - 1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Train")
plt.plot(val_loss, label="Validation")
plt.axvline(initial_epochs - 1, color="gray", linestyle="--", label="Fine-tuning starts")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 10 - Evaluate on the test set; confusion matrix; report

**What it does:**
- Evaluates on the held-out test batches. `y_true` is collected from the dataset; `y_pred` from the model's argmax.
- `test_ds` is built from fixed arrays and is never shuffled, so labels and predictions line up.

*Expected output (illustrative):* Test accuracy around 0.87 - 0.90 (expected from your PDF), a 5 x 5 confusion matrix and the precision/recall/F1 table.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print("Test accuracy:", round(test_acc, 4))

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Cell 11 - Show predictions on test images

**What it does:**
- `(x + 1) / 2` undoes the [-1, 1] scaling so the image can be displayed.
- Title shows the predicted class and its probability; green = correct, red = wrong.

*Expected output (illustrative):* 8 flower images with predicted class names and confidence values.

In [ ]:
images, labels = next(iter(test_ds))
probs = model.predict(images, verbose=0)
preds = np.argmax(probs, axis=1)

plt.figure(figsize=(12, 6))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow(((images[i].numpy() + 1) / 2).clip(0, 1))
    ok = preds[i] == labels[i].numpy()
    plt.title(class_names[preds[i]] + " (" + str(round(float(probs[i][preds[i]]), 2)) + ")",
              color="green" if ok else "red")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Result / Conclusion

Transfer learning with MobileNetV2 (feature extraction followed by fine-tuning of the top 30 layers) classified flowers with roughly 87-90% validation accuracy using little training time.